In [1]:
import pandas as pd
import sqlite3
import os

In [2]:
# Connect to the SQLite database we built during the ingestion stage
conn = sqlite3.connect("C:/Users/durel/Documents/Python Programming - Wfu/Personal Projects/aml-rule-engine/data/aml_engine.db")

In [7]:
df = pd.read_sql("SELECT * FROM transactions",conn)
df_filtered = pd.read_sql("SELECT * FROM transactions LIMIT 5", conn)
print(df_filtered.shape)
df_filtered.head()

(5, 11)


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,TRANSFER,181.00,C1305486145,181.0,0.0,C553264065,0.0,0.00,1,0
1,1,CASH_OUT,181.00,C840083671,181.0,0.0,C38997010,21182.0,0.00,1,0
2,1,CASH_OUT,229133.94,C905080434,15325.0,0.0,C476402209,5083.0,51513.44,0,0
3,1,TRANSFER,215310.30,C1670993182,705.0,0.0,C1100439041,22425.0,0.00,0,0
4,1,TRANSFER,311685.89,C1984094095,10835.0,0.0,C932583850,6267.0,2719172.89,0,0


# Structuring
First Rule: Structuring
Structuring is when someone breaks up a large amount into multiple smaller transactions to stay below the 10,000 reporting threshold. In PaySim, we don't have cash transactions, but we can detect the same pattern using TRANSFER and CASH_OUT transactions where a single sender makes multiple transactions in the same time window that each fall between 9,000 and 10,000.

In [11]:
def detect_structuring(df, window=1, min_amount=9000, max_amount=10000, min_count=2):
    """
    Flags senders who make multiple transactions in the same time window
    where each transaction falls in the $9,000 to $10,000 range.
    
    window: number of steps to group transactions into (1 step = 1 hour in PaySim)
    min_amount / max_amount: the structuring band we're watching
    min_count: minimum number of transactions in the band to trigger an alert
    """
    
    # Filter to only TRANSFER and CASH_OUT since those are the fraud-relevant types
    aml_types = ["TRANSFER", "CASH_OUT"]
    filtered = df[df["type"].isin(aml_types)].copy()
    
    # Keep only transactions that fall within the structuring band
    in_band = filtered[(filtered["amount"] >= min_amount) & (filtered["amount"] <= max_amount)].copy()
    
    # Group by sender and time window, then count how many transactions each sender has in the band within that window
    in_band["window"] = in_band["step"] // window
    
    grouped = (
        in_band.groupby(["nameOrig", "window"])
        .agg(
            transaction_count=("amount", "count"),  # how many transactions in the band
            total_amount=("amount", "sum"),          # total value across those transactions
            steps=("step", list)                     # which steps they occurred on
        )
        .reset_index()
    )
    
    # Only keep groups that hit the minimum transaction count threshold
    alerts = grouped[grouped["transaction_count"] >= min_count].copy()
    
    # Tag these rows so we know which rule fired
    alerts["rule"] = "STRUCTURING"
    
    return alerts

# Run the rule against the full dataset
structuring_alerts = detect_structuring(df)
print(f"Structuring alerts found: {len(structuring_alerts)}")
structuring_alerts.head()

Structuring alerts found: 0


,nameOrig,window,transaction_count,total_amount,steps,rule


In [9]:
# Check how many transactions actually fall in our structuring band
aml_types = ["TRANSFER", "CASH_OUT"]
filtered = df[df["type"].isin(aml_types)]

in_band = filtered[(filtered["amount"] >= 9000) & (filtered["amount"] <= 10000)]

print(f"Total AML-type transactions: {len(filtered)}")
print(f"Transactions in $9k-$10k band: {len(in_band)}")
print(f"Unique senders in band: {in_band['nameOrig'].nunique()}")

Total AML-type transactions: 2770409
Transactions in $9k-$10k band: 8024
Unique senders in band: 8024


In [10]:
# Check the actual transaction type distribution in our loaded DataFrame
print(df["type"].value_counts())

type
CASH_OUT    2237500
TRANSFER     532909
Name: count, dtype: int64


In [12]:
# Check how many transactions each in-band sender makes across the entire dataset
aml_types = ["TRANSFER", "CASH_OUT"]
filtered = df[df["type"].isin(aml_types)]

in_band = filtered[
    (filtered["amount"] >= 9000) &
    (filtered["amount"] <= 10000)
]

# For each sender who appeared in the band, count their total transactions
sender_counts = (
    filtered[filtered["nameOrig"].isin(in_band["nameOrig"])]
    .groupby("nameOrig")["amount"]
    .count()
    .value_counts()
    .head(10)
)

print(sender_counts)

amount
1    8018
2       6
Name: count, dtype: int64


In [13]:
# Re-run with wider amount band and 3-day (72 step) window
structuring_alerts = detect_structuring(df, window=72, min_amount=8000, max_amount=10000, min_count=2)
print(f"Structuring alerts found: {len(structuring_alerts)}")
structuring_alerts.head()

Structuring alerts found: 0


,nameOrig,window,transaction_count,total_amount,steps,rule


In [14]:
# Check sender transaction counts with the wider $8k-$10k band
in_band_wide = filtered[
    (filtered["amount"] >= 8000) &
    (filtered["amount"] <= 10000)
]

print(f"Transactions in wider band: {len(in_band_wide)}")
print(f"Unique senders: {in_band_wide['nameOrig'].nunique()}")

# Distribution of how many times each sender appears in the band
print(in_band_wide.groupby("nameOrig")["amount"].count().value_counts().head(10))

Transactions in wider band: 15857
Unique senders: 15857
amount
1    15857
Name: count, dtype: int64


PaySim senders almost never make more than one transaction within a narrow amount band. Rather than forcing a rule that produces no alerts, I'm going to try adapting the detection logic to a smurfing typology, which flags senders who make multiple sub-10,000 transactions to different destinations within a 72 hour window where the combined total exceeds $10,000. This approach is consistent with real AML detection frameworks and may better reflect what the data can actually support.

In [15]:
def detect_structuring(df, window=72, individual_max=10000, combined_min=10000, min_count=2):
    """
    Detects smurfing behavior: a sender making multiple sub-threshold transactions
    to different destinations within a time window where the combined total exceeds
    the reporting threshold.
    window: number of steps to group transactions into (72 steps = 3 days in PaySim)
    individual_max: each transaction must be below this amount (the structuring threshold)
    combined_min: the group of transactions must exceed this amount in total
    min_count: minimum number of transactions required to trigger an alert
    """
    
    # Filter to fraud-relevant transaction types confirmed during EDA
    aml_types = ["TRANSFER", "CASH_OUT"]
    filtered = df[df["type"].isin(aml_types)].copy()
    
    # Keep only transactions that are individually below the reporting threshold
    below_threshold = filtered[filtered["amount"] < individual_max].copy()
    
    # Assign each transaction to a time window bucket
    below_threshold["window"] = below_threshold["step"] // window
    
    # Group by sender and window, aggregating key metrics
    grouped = (
        below_threshold.groupby(["nameOrig", "window"])
        .agg(
            transaction_count=("amount", "count"),    # number of transactions in window
            total_amount=("amount", "sum"),            # combined value across transactions
            unique_destinations=("nameDest", "nunique"), # number of different recipients
            steps=("step", list)                       # which steps the transactions occurred on
        )
        .reset_index()
    )
    
    # Apply all three conditions to flag genuine smurfing behavior
    alerts = grouped[
        (grouped["transaction_count"] >= min_count) &      # multiple transactions
        (grouped["total_amount"] >= combined_min) &         # combined total exceeds threshold
        (grouped["unique_destinations"] > 1)                # money going to different places
    ].copy()
    
    # Tag with rule name for the alerts DataFrame we'll build later
    alerts["rule"] = "STRUCTURING"
    
    return alerts

# Run the updated rule
structuring_alerts = detect_structuring(df)
print(f"Structuring alerts found: {len(structuring_alerts)}")
structuring_alerts.head()

Structuring alerts found: 0


,nameOrig,window,transaction_count,total_amount,unique_destinations,steps,rule


In [16]:
# Check how many senders make more than one sub-$10k transaction at all
aml_types = ["TRANSFER", "CASH_OUT"]
filtered = df[df["type"].isin(aml_types)].copy()

below_threshold = filtered[filtered["amount"] < 10000].copy()

sender_counts = below_threshold.groupby("nameOrig")["amount"].count()

print(f"Total sub-$10k transactions: {len(below_threshold)}")
print(f"Unique senders: {sender_counts.nunique()}")
print(f"Senders with more than 1 transaction: {(sender_counts > 1).sum()}")

Total sub-$10k transactions: 78841
Unique senders: 2
Senders with more than 1 transaction: 2


In [17]:
# Look at those 2 senders directly
repeat_senders = sender_counts[sender_counts > 1].index.tolist()

print(below_threshold[below_threshold["nameOrig"].isin(repeat_senders)][
    ["step", "nameOrig", "nameDest", "amount", "type"]
].sort_values(["nameOrig", "step"]))

         step     nameOrig     nameDest   amount      type
1682162   282  C1623366178  C2044101258  6419.01  CASH_OUT
2134754   348  C1623366178  C1606360728  3330.59  CASH_OUT
754231    160    C94830298  C1446481181  5070.82  CASH_OUT
2507805   399    C94830298  C1983933473   595.94  CASH_OUT


Based on my analysis above, The structuring rule logic is finalized but produces minimal alerts on PaySim due to the dataset limitation. In a production environment with real banking data, this rule would be expected to generate meaningful alert volume. I am documenting this limitation here rather than artificially tuning parameters to force results.

# Velocity 95th percentile

In [18]:
def detect_velocity_spike(df, percentile=95, min_count=2):
    """
    Flags senders whose total transaction amount exceeds the population-level
    95th percentile threshold and who made more than one transaction.
    percentile: the population threshold to flag against
    min_count: sender must have more than this many transactions to be flagged
    """
    # Filter to fraud-relevant transaction types
    aml_types = ["TRANSFER", "CASH_OUT"]
    filtered = df[df["type"].isin(aml_types)].copy()
    
    # Aggregate total amount and transaction count per sender
    sender_agg = (filtered.groupby("nameOrig").agg(total_amount=("amount", "sum"),transaction_count=("amount", "count")).reset_index())
    
    # Calculate the population-level threshold
    threshold = sender_agg["total_amount"].quantile(percentile / 100)
    print(f"95th percentile threshold: ${threshold:,.2f}")
    
    # Flag senders who exceed the threshold and have multiple transactions
    alerts = sender_agg[(sender_agg["total_amount"] >= threshold) & (sender_agg["transaction_count"] >= min_count)].copy()
    
    alerts["rule"] = "VELOCITY_SPIKE"
    
    return alerts

velocity_alerts = detect_velocity_spike(df)
print(f"Velocity spike alerts found: {len(velocity_alerts)}")
velocity_alerts.head()

95th percentile threshold: $955,791.02
Velocity spike alerts found: 228


,nameOrig,total_amount,transaction_count,rule
19534,C101359786,968812.14,2,VELOCITY_SPIKE
24249,C1016949303,1250604.14,2,VELOCITY_SPIKE
31351,C1021950674,1154220.93,2,VELOCITY_SPIKE
31475,C1022036948,3938347.98,2,VELOCITY_SPIKE
41162,C1028766280,3858106.20,2,VELOCITY_SPIKE


My original plan was to compare each sender against their own transaction history, flag anyone who suddenly spiked above their personal baseline. That felt intuitive. The problem is that when I actually looked at the data, the vast majority of senders in PaySim make exactly one transaction ever, so there is no baseline to compare against.

I ended up pivoting to a population level approach instead. I aggregated the total amount sent per sender and flagged anyone sitting above the 95th percentile who also made more than one transaction. The threshold came out to just under $956,000 which honestly feels like a reasonable number to raise an eyebrow at. 228 senders crossed that line, and that felt like a meaningful result rather than something I tuned until it looked good.

In a real system with richer transaction history this rule would probably use peer group segmentation, but for what PaySim gives us I think this holds up.

# Dormant Account Activity
1. Find each account's first and last transaction step
2. Calculate the gap between them
3. Flag accounts where the gap exceeds a threshold and the first transaction after the dormant period is large

In [19]:
def detect_dormant_activity(df, min_gap=30, min_reactivation_amount=50000):
    """
    Flags sender accounts that were dormant for an extended period and then
    reactivated with a large transaction.
    min_gap: minimum number of steps between first and last transaction to
             consider an account dormant (30 steps = 30 hours in PaySim)
    min_reactivation_amount: the first transaction after reactivation must
                             exceed this amount to trigger an alert
    """
    
    # Filter to fraud-relevant transaction types
    aml_types = ["TRANSFER", "CASH_OUT"]
    filtered = df[df["type"].isin(aml_types)].copy()
    
    # For each sender, find their first and last transaction step
    sender_activity = (
        filtered.groupby("nameOrig")
        .agg(
            first_step=("step", "min"),   # when they first appeared
            last_step=("step", "max"),    # when they last appeared
            transaction_count=("amount", "count")
        )
        .reset_index()
    )
    
    # Calculate the gap between first and last transaction
    sender_activity["step_gap"] = (sender_activity["last_step"] - sender_activity["first_step"])
    
    # Find the amount of each sender's first transaction
    # This represents the reactivation event after dormancy
    first_transactions = (
        filtered.sort_values("step")
        .groupby("nameOrig")
        .first()
        .reset_index()[["nameOrig", "amount"]]
        .rename(columns={"amount": "first_transaction_amount"})
    )
    
    # Merge the reactivation amount back onto sender activity
    sender_activity = sender_activity.merge(first_transactions, on="nameOrig")
    
    # Flag senders who meet both dormancy and reactivation conditions
    alerts = sender_activity[(sender_activity["step_gap"] >= min_gap) & (sender_activity["first_transaction_amount"] >= min_reactivation_amount)].copy()
    
    alerts["rule"] = "DORMANT_ACTIVITY"
    
    return alerts

dormant_alerts = detect_dormant_activity(df)
print(f"Dormant activity alerts found: {len(dormant_alerts)}")
dormant_alerts.head()

Dormant activity alerts found: 1261


,nameOrig,first_step,last_step,transaction_count,step_gap,first_transaction_amount,rule
2885,C1002018924,301,358,2,57,217695.29,DORMANT_ACTIVITY
5429,C10037435,235,402,2,167,392937.78,DORMANT_ACTIVITY
7117,C1004939913,9,259,2,250,733478.62,DORMANT_ACTIVITY
8110,C1005646800,21,184,2,163,156163.47,DORMANT_ACTIVITY
9105,C1006323448,19,251,2,232,356995.42,DORMANT_ACTIVITY


This one came together pretty cleanly compared to the others. This is a simple idea honestly, an account that sits quiet for a long time and then wakes up with a large transaction is worth looking at. I defined dormancy as a gap of at least 30 steps between first and last transaction, and set the reactivation threshold at $50,000 since anything lower felt like it would just be noise.

1,261 senders triggered this rule which does feel reasonable. Looking at the output you can see step gaps in the hundreds and first transaction amounts well into the six figures, so the rule is clearly catching something real rather than just random activity. In a production system you would probably layer in account age and customer risk rating on top of this, but standalone, I think it holds up well.

# Rapid Movement
1. A sender sends money to a destination
2. That destination then quickly becomes a sender and moves the money again
3. That pattern of money hopping between accounts reflects layering behavior in real AML cases

In [20]:
def detect_rapid_movement(df, max_step_gap=24):
    """
    Detects two-hop transaction chaining where money moves from account A to B,
    then B quickly sends money onward to C. This reflects layering behavior
    where funds are moved through intermediary accounts to obscure the origin.
    
    max_step_gap: maximum number of steps allowed between hop 1 and hop 2
                  (24 steps = 24 hours in PaySim)
    """
    
    # Filter to fraud-relevant transaction types
    aml_types = ["TRANSFER", "CASH_OUT"]
    filtered = df[df["type"].isin(aml_types)].copy()
    
    # Hop 1: all outbound transactions (A sends to B)
    hop1 = filtered[["step", "nameOrig", "nameDest", "amount"]].copy()
    hop1.columns = ["step_1", "account_a", "account_b", "amount_1"]
    
    # Hop 2: all outbound transactions (B sends to C)
    hop2 = filtered[["step", "nameOrig", "nameDest", "amount"]].copy()
    hop2.columns = ["step_2", "account_b", "account_c", "amount_2"]
    
    # Join on account_b where B appears as both destination and sender
    # This is the key step that identifies the intermediary account
    chained = hop1.merge(hop2, on="account_b")
    
    # Keep only cases where hop 2 happens AFTER hop 1
    chained = chained[chained["step_2"] > chained["step_1"]].copy()
    
    # Keep only cases where the two hops happen within our time window
    chained["step_gap"] = chained["step_2"] - chained["step_1"]
    chained = chained[chained["step_gap"] <= max_step_gap].copy()
    
    # Drop cases where money is just going back to the original sender
    chained = chained[chained["account_c"] != chained["account_a"]].copy()
    
    chained["rule"] = "RAPID_MOVEMENT"
    
    return chained

rapid_alerts = detect_rapid_movement(df)
print(f"Rapid movement alerts found: {len(rapid_alerts)}")
rapid_alerts.head()

Rapid movement alerts found: 190


,step_1,account_a,account_b,amount_1,step_2,account_c,amount_2,step_gap,rule
151,14,C162362674,C1939958280,9485.21,37,C232785081,400729.53,23,RAPID_MOVEMENT
154,14,C1209523957,C1939958280,160046.07,37,C232785081,400729.53,23,RAPID_MOVEMENT
195,16,C1810325871,C1939958280,420986.32,37,C232785081,400729.53,21,RAPID_MOVEMENT
213,16,C696809778,C585981957,227772.92,40,C1871172330,484468.34,24,RAPID_MOVEMENT
277,18,C523379204,C1136437763,50961.10,40,C495624128,170658.65,22,RAPID_MOVEMENT


Rapid movement was the rule I had to rethink the most. My first thought was to flag senders making multiple transactions within a single hour, but the data showed that PaySim senders almost never do that.

I pivoted to transaction chaining instead. The idea is to find cases where account A sends money to account B, and then account B turns around and sends money to a completely different account C within 24 hours. That intermediary hop is what makes it interesting from an AML perspective, it looks like layering where someone is trying to put distance between the origin and destination of funds.

190 alerts came out of this which feels right for a two hop chain. In a real system you would probably extend this to three or four hops and weight alerts by the speed and size of the movement, but I think two hops is a good starting starting point.

In [24]:
# Quick integration test to confirm src/rules.py is working correctly
# This imports from src/ the same way alerts.py will
import sys
sys.path.append("..") # adds the project root to the path so src/ is findable

from src.rules import (
    detect_structuring,
    detect_velocity_spike,
    detect_dormant_activity,
    detect_rapid_movement
)

# Run all four rules and print alert counts
print(f"Structuring:      {len(detect_structuring(df))} alerts")
print(f"Velocity Spike:   {len(detect_velocity_spike(df))} alerts")
print(f"Dormant Activity: {len(detect_dormant_activity(df))} alerts")
print(f"Rapid Movement:   {len(detect_rapid_movement(df))} alerts")

Structuring:      0 alerts
Velocity Spike:   228 alerts
Dormant Activity: 1261 alerts
Rapid Movement:   190 alerts


In [25]:
# Test the full alerts pipeline from src/alerts.py
from src.alerts import run_alert_pipeline

# Run the pipeline and save the CSV
alerts = run_alert_pipeline(df)

# Preview the combined alerts DataFrame
print(alerts["rule"].value_counts())
alerts.head(10)

Alerts saved to outputs/alerts/alerts.csv
Total alerts: 1679
rule
DORMANT_ACTIVITY    1261
VELOCITY_SPIKE       228
RAPID_MOVEMENT       190
Name: count, dtype: int64


,account_id,rule,alert_amount,alert_details,transaction_count
0,C101359786,VELOCITY_SPIKE,968812.14,Sender total of $968812.14 exceeds the 95th pe...,2
1,C1016949303,VELOCITY_SPIKE,1250604.14,Sender total of $1250604.14 exceeds the 95th p...,2
2,C1021950674,VELOCITY_SPIKE,1154220.93,Sender total of $1154220.93 exceeds the 95th p...,2
3,C1022036948,VELOCITY_SPIKE,3938347.98,Sender total of $3938347.98 exceeds the 95th p...,2
4,C1028766280,VELOCITY_SPIKE,3858106.20,Sender total of $3858106.2 exceeds the 95th pe...,2
5,C1029713918,VELOCITY_SPIKE,2635985.86,Sender total of $2635985.86 exceeds the 95th p...,2
6,C1040334487,VELOCITY_SPIKE,1073382.12,Sender total of $1073382.12 exceeds the 95th p...,2
7,C1068240459,VELOCITY_SPIKE,1340257.44,Sender total of $1340257.44 exceeds the 95th p...,2
8,C1095346997,VELOCITY_SPIKE,1298920.62,Sender total of $1298920.62 exceeds the 95th p...,2
9,C1101705604,VELOCITY_SPIKE,5414515.92,Sender total of $5414515.92 exceeds the 95th p...,2
